<style>
.jp-RenderedHTMLCommon table,
.rendered_html table {
  margin-left: 0 !important;
  margin-right: auto !important;
}
.jp-RenderedHTMLCommon th,
.jp-RenderedHTMLCommon td,
.rendered_html th,
.rendered_html td {
  text-align: left !important;
}
</style>

# 06 - Scaling, Normalization, and Encoding

<div style="background:#F7FAFC; border:1px solid #D9E2EC; border-left:6px solid #2F80ED; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/goal.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Notebook Header</b><br>
<b>Day:</b> Day 1 - EDA and preprocessing foundations<br>
<b>Difficulty:</b> Beginner<br>
<b>Estimated time:</b> 60-75 minutes<br>
<b>Prerequisites:</b> Notebook 05 completed<br>
<b>Output:</b> You will transform numeric and categorical columns into analysis-ready formats.<br>
<b>Next notebook:</b> Day 1 mini EDA project
</div>

<details>
<summary><b>Instructor talk track</b></summary>

This notebook is about preparing columns. Keep the explanation practical: numeric columns can be on very different scales, and text categories need to be converted into numeric form for many later workflows. Do not rush the difference between standardization and normalization.

</details>

## <img src="../../../assets/icons/map.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Learning Map

By the end of this notebook, you should be able to:

- Explain why numeric columns sometimes need scaling
- Apply standard scaling
- Apply min-max normalization
- Explain why categorical columns need encoding
- Apply one-hot encoding
- Apply simple ordered mapping
- Save a preprocessing-ready dataset

## <img src="../../../assets/icons/loop.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Signature Learning Loop

```text
QUESTION -> DATA -> CODE -> EVIDENCE -> DECISION
```


## <img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Concept 1: Why Preprocessing Matters

<div style="background:#EAF3FF; border-left:6px solid #2F80ED; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Concept</b><br>
Preprocessing means converting data into a cleaner and more useful format for analysis.
</div>

### Two common preprocessing needs

- Numeric columns may use very different scales.
- Text categories may need to become numeric columns.

Example:

- `monthly_spend` may range from hundreds to thousands.
- `visits_per_month` may range from 1 to 6.
- `membership` is text such as Gold, Silver, and Bronze.

<div style="background:#E9F8EF; border-left:6px solid #27AE60; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/code.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Code Lab</b><br>
Import libraries and load the cleaned customer dataset.
</div>


In [1]:
from pathlib import Path
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, StandardScaler

current = Path.cwd().resolve()
project_root = current

for candidate in [current, *current.parents]:
    if candidate.name == "applied_ds_ml":
        project_root = candidate
        break

data_path = project_root / "data" / "customer_activity_clean.csv"

if not data_path.exists():
    raise FileNotFoundError("Run Notebook 03 first so data/customer_activity_clean.csv is created.")

customers = pd.read_csv(data_path)
customers


,customer_id,name,city,membership,monthly_spend,visits_per_month,signup_month
0,101,Asha,Mumbai,Gold,1200.0,4.0,Jan
1,102,Ravi,Delhi,Silver,850.0,3.0,Feb
2,103,Meera,Chennai,Gold,1450.0,5.0,Feb
3,104,John,Mumbai,Bronze,700.0,2.0,Mar
4,105,Fatima,Unknown,Silver,1025.0,6.0,Apr
5,106,Chen,Chennai,Gold,1600.0,5.0,Apr
6,107,Sara,Delhi,Unknown,950.0,4.0,May
7,108,Vikram,Mumbai,Bronze,1025.0,1.0,Jun
8,109,Nina,Delhi,Silver,1100.0,4.0,Jun


## <img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Concept 2: Compare Column Scales

<div style="background:#EAF3FF; border-left:6px solid #2F80ED; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Concept</b><br>
Scale means the size range of values in a column.
</div>

### Why scale matters

If one column has large values and another column has small values, the large-valued column can dominate some calculations later.

<div style="background:#E9F8EF; border-left:6px solid #27AE60; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/code.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Code Lab</b><br>
Compare basic summaries of numeric columns.
</div>


In [2]:
numeric_columns = ["monthly_spend", "visits_per_month"]

customers[numeric_columns].describe().round(2)


,monthly_spend,visits_per_month
count,9.00,9.00
mean,1100.00,3.78
std,282.57,1.56
min,700.00,1.00
25%,950.00,3.00
50%,1025.00,4.00
75%,1200.00,5.00
max,1600.00,6.00


## <img src="../../../assets/icons/read-output.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Read the Output: Column Scales

- `monthly_spend` has larger values.
- `visits_per_month` has smaller values.
- Scaling helps put numeric columns into more comparable forms.


## <img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Concept 3: Standard Scaling

<div style="background:#EAF3FF; border-left:6px solid #2F80ED; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Concept</b><br>
Standard scaling changes values so the column has mean near 0 and standard deviation near 1.
</div>

### Simple meaning

Standard scaling answers:

- Is this value above or below average?
- How far is it from average in standard-deviation units?

### Formula idea

```text
standard scaled value = (value - mean) / standard deviation
```

<div style="background:#E9F8EF; border-left:6px solid #27AE60; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/code.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Code Lab</b><br>
Apply standard scaling to numeric columns.
</div>


In [3]:
standard_scaler = StandardScaler()
standard_scaled_values = standard_scaler.fit_transform(customers[numeric_columns])

standard_scaled_df = pd.DataFrame(
    standard_scaled_values,
    columns=["monthly_spend_standard_scaled", "visits_per_month_standard_scaled"]
)

standard_scaled_df.round(3)


,monthly_spend_standard_scaled,visits_per_month_standard_scaled
0,0.375,0.151
1,-0.938,-0.528
2,1.314,0.829
3,-1.501,-1.206
4,-0.282,1.508
5,1.877,0.829
6,-0.563,0.151
7,-0.282,-1.884
8,0.000,0.151


## <img src="../../../assets/icons/read-output.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Read the Output: Standard Scaling

- Values near `0` are close to average.
- Positive values are above average.
- Negative values are below average.
- Larger absolute values are farther from average.


## <img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Concept 4: Check Standard Scaled Summary

<div style="background:#EAF3FF; border-left:6px solid #2F80ED; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Concept</b><br>
After standard scaling, the mean should be close to 0 and the standard deviation should be close to 1.
</div>

<div style="background:#E9F8EF; border-left:6px solid #27AE60; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/code.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Code Lab</b><br>
Summarize the standard-scaled columns.
</div>


In [4]:
standard_scaled_df.describe().round(3)


,monthly_spend_standard_scaled,visits_per_month_standard_scaled
count,9.000,9.000
mean,-0.000,0.000
std,1.061,1.061
min,-1.501,-1.884
25%,-0.563,-0.528
50%,-0.282,0.151
75%,0.375,0.829
max,1.877,1.508


## <img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Concept 5: Min-Max Normalization

<div style="background:#EAF3FF; border-left:6px solid #2F80ED; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Concept</b><br>
Min-max normalization changes values to a fixed range, usually 0 to 1.
</div>

### Simple meaning

- `0` means the smallest value in that column.
- `1` means the largest value in that column.
- Values between 0 and 1 show relative position.

### Formula idea

```text
normalized value = (value - minimum) / (maximum - minimum)
```

<div style="background:#E9F8EF; border-left:6px solid #27AE60; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/code.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Code Lab</b><br>
Apply min-max normalization to numeric columns.
</div>


In [5]:
minmax_scaler = MinMaxScaler()
normalized_values = minmax_scaler.fit_transform(customers[numeric_columns])

normalized_df = pd.DataFrame(
    normalized_values,
    columns=["monthly_spend_normalized", "visits_per_month_normalized"]
)

normalized_df.round(3)


,monthly_spend_normalized,visits_per_month_normalized
0,0.556,0.6
1,0.167,0.4
2,0.833,0.8
3,0.000,0.2
4,0.361,1.0
5,1.000,0.8
6,0.278,0.6
7,0.361,0.0
8,0.444,0.6


## <img src="../../../assets/icons/read-output.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Read the Output: Normalization

- The smallest value becomes `0`.
- The largest value becomes `1`.
- Other values are placed between `0` and `1`.


## <img src="../../../assets/icons/recap.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Scaling vs Normalization

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
  <thead>
    <tr>
      <th style="text-align:left; padding:6px 12px;">Method</th>
      <th style="text-align:left; padding:6px 12px;">Result</th>
      <th style="text-align:left; padding:6px 12px;">Simple meaning</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:left; padding:6px 12px;">Standard scaling</td><td style="text-align:left; padding:6px 12px;">Mean near 0, standard deviation near 1</td><td style="text-align:left; padding:6px 12px;">How far from average?</td></tr>
    <tr><td style="text-align:left; padding:6px 12px;">Min-max normalization</td><td style="text-align:left; padding:6px 12px;">Values between 0 and 1</td><td style="text-align:left; padding:6px 12px;">Where is value between min and max?</td></tr>
  </tbody>
</table>


## <img src="../../../assets/icons/practice.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Checkpoint 1: Numeric Transformations

Answer these before continuing:

1. What does standard scaling do?
2. What does min-max normalization do?
3. Which method creates values between `0` and `1`?
4. Which method helps explain distance from average?

<div style="background:#FFF8E1; border-left:6px solid #F2C94C; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/read-output.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Expected thinking</b><br>
Both methods change numeric values, but they answer different practical questions.
</div>


## <img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Concept 6: Why Encoding Is Needed

<div style="background:#EAF3FF; border-left:6px solid #2F80ED; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Concept</b><br>
Encoding means converting text categories into numeric columns.
</div>

### Why encode categories?

Many later data workflows need numbers. Text values like `Gold`, `Silver`, and `Bronze` must be represented in numeric form.

<div style="background:#E9F8EF; border-left:6px solid #27AE60; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/code.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Code Lab</b><br>
Inspect categorical columns before encoding.
</div>


In [ ]:
categorical_columns = ["city", "membership", "signup_month"]

for column in categorical_columns:
    print(column)
    print(customers[column].unique())
    print()


## <img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Concept 7: One-Hot Encoding

<div style="background:#EAF3FF; border-left:6px solid #2F80ED; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Concept</b><br>
One-hot encoding creates separate 0/1 columns for each category.
</div>

### Example

If `city` contains `Mumbai`, `Delhi`, and `Chennai`, one-hot encoding creates columns like:

- `city_Mumbai`
- `city_Delhi`
- `city_Chennai`

A row gets `1` for the matching city and `0` for the others.

<div style="background:#E9F8EF; border-left:6px solid #27AE60; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/code.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Code Lab</b><br>
One-hot encode the `city` column.
</div>


In [ ]:
city_encoded = pd.get_dummies(customers["city"], prefix="city", dtype=int)

city_encoded


## <img src="../../../assets/icons/read-output.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Read the Output: One-Hot Encoding

- Each city became a separate column.
- `1` means the row belongs to that city.
- `0` means the row does not belong to that city.


## <img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Concept 8: Encode Multiple Columns

<div style="background:#EAF3FF; border-left:6px solid #2F80ED; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Concept</b><br>
We can one-hot encode multiple categorical columns at the same time.
</div>

<div style="background:#E9F8EF; border-left:6px solid #27AE60; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/code.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Code Lab</b><br>
One-hot encode `city` and `membership` together.
</div>


In [ ]:
encoded_categories = pd.get_dummies(customers[["city", "membership"]], dtype=int)

encoded_categories


## <img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Concept 9: Ordered Mapping

<div style="background:#EAF3FF; border-left:6px solid #2F80ED; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Concept</b><br>
Ordered mapping converts categories to numbers when the categories have a meaningful order.
</div>

### Example order

For membership:

```text
Bronze < Silver < Gold
```

So we can map:

```text
Bronze = 1, Silver = 2, Gold = 3
```

<div style="background:#FFECEC; border-left:6px solid #EB5757; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/caution.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Caution</b><br>
Use ordered mapping only when the categories truly have an order.
</div>

<div style="background:#E9F8EF; border-left:6px solid #27AE60; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/code.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Code Lab</b><br>
Create a membership level column.
</div>


In [ ]:
membership_order = {
    "Bronze": 1,
    "Silver": 2,
    "Gold": 3,
    "Unknown": 0,
}

customers["membership_level"] = customers["membership"].map(membership_order)

customers[["membership", "membership_level"]]


## <img src="../../../assets/icons/recap.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Encoding Summary Table

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
  <thead>
    <tr>
      <th style="text-align:left; padding:6px 12px;">Method</th>
      <th style="text-align:left; padding:6px 12px;">Use when</th>
      <th style="text-align:left; padding:6px 12px;">Example</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:left; padding:6px 12px;">One-hot encoding</td><td style="text-align:left; padding:6px 12px;">Categories have no natural order</td><td style="text-align:left; padding:6px 12px;">City</td></tr>
    <tr><td style="text-align:left; padding:6px 12px;">Ordered mapping</td><td style="text-align:left; padding:6px 12px;">Categories have a meaningful order</td><td style="text-align:left; padding:6px 12px;">Membership level</td></tr>
  </tbody>
</table>


## <img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Concept 10: Create a Preprocessing-Ready Table

<div style="background:#EAF3FF; border-left:6px solid #2F80ED; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Concept</b><br>
A preprocessing-ready table combines useful original columns with transformed columns.
</div>

### What we will include

- Customer identity columns
- Original categorical columns for readability
- Scaled numeric columns
- Encoded categorical columns

<div style="background:#E9F8EF; border-left:6px solid #27AE60; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/code.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Code Lab</b><br>
Build a preprocessing-ready DataFrame.
</div>


In [ ]:
readable_columns = customers[["customer_id", "name", "city", "membership", "signup_month"]]

preprocessed_customers = pd.concat(
    [
        readable_columns.reset_index(drop=True),
        standard_scaled_df.reset_index(drop=True),
        normalized_df.reset_index(drop=True),
        encoded_categories.reset_index(drop=True),
        customers[["membership_level"]].reset_index(drop=True),
    ],
    axis=1
)

preprocessed_customers


## <img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Concept 11: Save the Preprocessed Dataset

<div style="background:#EAF3FF; border-left:6px solid #2F80ED; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Concept</b><br>
Saving transformed data lets later notebooks reuse the same prepared table.
</div>

<div style="background:#E9F8EF; border-left:6px solid #27AE60; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/code.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Code Lab</b><br>
Save the preprocessed dataset.
</div>


In [ ]:
output_path = project_root / "data" / "customer_activity_preprocessed.csv"
preprocessed_customers.to_csv(output_path, index=False)

print("Saved preprocessed dataset to:", output_path)
print("Rows and columns:", preprocessed_customers.shape)


## <img src="../../../assets/icons/caution.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Common Mistakes to Avoid

<div style="background:#FFECEC; border-left:6px solid #EB5757; padding:14px; border-radius:6px;">
<b>Read this before practice.</b>
</div>

- Thinking standard scaling and normalization are the same.
- Forgetting that standard-scaled values can be negative.
- Expecting normalized values to be below 0 or above 1.
- Using ordered mapping for categories that do not have a real order.
- Deleting original readable columns too early.
- Forgetting to save the transformed dataset.


## <img src="../../../assets/icons/practice.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Practice Task

Complete these tasks:

1. Standard scale only `monthly_spend` and inspect the result.
2. Normalize only `visits_per_month` and inspect the result.
3. One-hot encode `signup_month`.
4. Explain why `city` should use one-hot encoding instead of ordered mapping.
5. Add the encoded `signup_month` columns to `preprocessed_customers`.

## <img src="../../../assets/icons/practice.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Exit Ticket

Answer these before closing the notebook:

1. What does standard scaling do?
2. What does min-max normalization do?
3. What is one-hot encoding?
4. When is ordered mapping acceptable?
5. Why should we keep readable columns while learning?

## <img src="../../../assets/icons/recap.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Ready for the Next Notebook

You are ready for the next notebook if you can:

- Compare numeric column scales.
- Apply standard scaling.
- Apply min-max normalization.
- One-hot encode categorical columns.
- Use ordered mapping only for ordered categories.
- Save a preprocessed dataset.
